In [1]:
# import dependencies
from matplotlib import pyplot as plt 
from seaborn import color_palette
import pandas as pd
import numpy as np

from kinetics.functions import compute_initial_reaction_slope_fast, fit_and_plot_michaelis_menten
from scipy.stats import linregress

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Processing data from a Michaelis-Menten curve generation experiment

This outlines how to fit steady-state kinetic constants to data from a Michaelis-Menten curve generation plate reader experiment.

The example data here was generated a NADH standard curve experiment and 


## 1. Load standard curve data

Different plate readers will output data in different formats. Before starting analysis, use a program like Excel to reformat the data as a .csv file such that each column contains measurements for each sample. If your experiment contains replicate samples, ensure that replicates are grouped in adjacent columns. Following these formatting conventions will make it straightforward to load and manipulate the data in Python. 

The example data here is from an NADH standard curve containing two replicates for each standard. Although not strictly necessary, I calculate an average over many measurements for each standard to control for the variability in the instrument readings. 

In [ ]:
# use pandas to read data
standards = pd.read_csv('./standard_curve.csv')

# if raw flourescence exceeds plate reader's dynamic range, will output OVRFLW string
# to make sure this doesn't cause issues, replace with nan
standards = standards.replace({'OVRFLW': np.nan})

# convert data into a numpy array and transpose for convenience
standards = standards.to_numpy().astype(float).T

# for each sample, calculate average fluorescence across each measurement
standards = np.nanmean(standards, axis=1)

# separate replicates into separate arrays via slicing
standards_r1, standards_r2 = standards[0::2], standards[1::2]

# calculate a mean and standard deviation for each standard across replicates
mean, std = np.nanmean(np.vstack([standards_r1, standards_r2]), axis=0), np.nanstd(np.vstack([standards_r1, standards_r2]), axis=0)

## 2. Fit standard curve data

As the experimentalist, you should know the concentration of standard in each well of your plate. With this information, create a numpy array 

In [ ]:
# the concentration of each standard in each column 
# units in this case are µM
standard_concentrations = np.array([2000, 1000, 500, 250, 125, 200, 100, 50, 25, 12.5])

# NADH standard curve becomes nonlinear at concentrations above 500µM
# we will exclude these from the fit using array masking
linear_mask = standard_concentrations < 500

# exclude nan values from fit using array masking
nan_mask = ~np.isnan(mean)

# fit a line to the standard data
fit = linregress(standard_concentrations[linear_mask & nan_mask], mean[linear_mask & nan_mask])

# plot the standard cuarve
fig, ax = plt.subplots()
ax.scatter(standard_concentrations, mean, color='black')
ax.errorbar(standard_concentrations, mean, yerr=std, fmt='o', color='black')
ax.plot(standard_concentrations[linear_mask], (standard_concentrations[linear_mask] * fit.slope) + fit.intercept, color='blue')
textstr = '\n'.join([
    r'$R^{2}=%.3f$' % (fit.rvalue)
])
ax.text(0.5, 0.5, textstr, transform=ax.transAxes)
ax.set_xlabel('[NADH] (µM)')
ax.set_ylabel('NADH Fluorescence')

## 3. Load progress curve data

As with the standard curve data, the progress curve data is formatted as a .csv file with columns containing timepoints for each sample. The very first column is the time, in units of seconds for this particular experiment.

In [4]:
data = pd.read_csv('./progress_curves.csv') # use pandas to read data
data = data.replace({'OVRFLW': np.nan})
data = data.to_numpy().astype(float).T # convert data into a numpy array
time, progress_curves = data[0], data[1:]

# convert progress curves to µM of NADH using standard curve from above
progress_curves = (progress_curves - fit.intercept) / fit.slope

## 4. Fit initial rates to progress curves

There are a few approaches one can use to fit initial rates to progress curves. Start with strategy three and experiment with others if the fits to the data look poor by eye. The optimal approach will vary depending on your data.

1. Fit a line to datapoints from the first 10-20% of the reaction.
2. Fit a single exponential model to the data. Then, evaluate the time derivative of the model at time zero to obtain the initial rate.
3.  

Always inspect the fits to the raw data by eye! If there are scoops or jumps that are artifactual, you may consider removing these points from the fit using array masking, as with the standard curve example above. 

In [5]:
v0, intercepts, rvalues, fit_indices = [], [], [], []
for curve in progress_curves:
    slope, intercept, rvalue, indices = compute_initial_reaction_slope_fast(time, curve, min_included_percent=10)
    v0.append(slope[0]), intercepts.append(intercept) ,rvalues.append(rvalue[0]), fit_indices.append(indices)
v0 = np.array(v0)

In [ ]:
# manually increment this variable to view data 
index = 10

fig, ax = plt.subplots()
ax.scatter(time, progress_curves[index], color='black', s=15)

x = np.linspace(*time[fit_indices[index]])
ax.plot(x, (x * v0[index]) + intercepts[index], color='blue')

ax.set_xlabel('Time (s)')
ax.set_ylabel('NADH (µM)')
textstr = '\n'.join([
    r'$R^{2}=%.3f$' % (rvalues[index])
])
ax.text(0.7, 0.5, textstr, transform=ax.transAxes)

## 5. Fit initial rates to background reactions

In scenarios where there is significant background signal, one can conduct control reactions without enzyme. Initial rates can then be fit to these control reactions. These rates represent the velocity of the background reaction and can be subtracted from the rates of the enzymatic reaction determined above. 

For this example, the background is minimal. Therefore, we will skip this step. 

## 6. Fit Michaelis-Menten model to initial rate data

Under the hood, `fit_and_plot_michaelis_menten` firsts subtracts backgorund rates if a `background_rates` keyword argument is provided. Then, it uses a curve fitting function implemented in `scipy.optimize.minimize` to fit steady-state kinetic constants to the data.

In [ ]:
# multiply v0 by -1 to obtain the rate of product production
# the assay used in this example couples product production to NADH depletion
v0 = v0 * -1

# divide v0 by the stoichiometry coefficient of the product
# doesn't matter in this example, since the stoichiometry coefficient is 1 
v0 = v0 / 1

# split 
v0_r1, v0_r2 = v0[0::2], v0[1::2]

# define concentrations of substrate in µM
substrate_concentrations = np.array([25, 12.5, 6.25, 3.125, 2.5, 1.25, 0.625, 0.3125])

# define concentration of enzyme in µM
enzyme_concentration = 25e-3 

# fit kcat and km to the data
fit_and_plot_michaelis_menten(
    rep_1_slopes=v0_r1, 
    rep_2_slopes=v0_r2,
    sub_concs=substrate_concentrations,
    e_conc=enzyme_concentration,
    conc_units='µM',
    title='Example MM Kinetics',
    background_rates=None # no background rates 
    )